# Per-tissue-type evaluation

Accuracy is the weighted Hungarian-aligned accuracy over all tiles from that slide, pooled across batches.

In [ ]:
import json, pathlib, pandas as pd

# ── slide_id → (tissue_type, species) mapping ──────────────────────
SLIDE_META = {
    "177_Human_Appendix_-_d0dcc529":    ("Appendix",         "Human"),
    "425_Human_Ovari_-_20_7ee9f54b":    ("Ovary",            "Human"),
    "76_HO135E_Compact_Bo_84b1d03b":    ("Compact bone",     "Human"),
    "93 Muscle T_S_ H_E - 2012-08-07":  ("Skeletal muscle",  "Human"),
    "190_Human_Pancreas_1_138151f1":     ("Pancreas",         "Human"),
    "215-1_Rat_Kidney_Car_9a27ed5a":     ("Kidney",           "Rat"),
    "396_Human_Prostate_3_d1b5a330":     ("Prostate",         "Human"),
    "434_Human_Cervix_434_12bce85a":     ("Cervix",           "Human"),
    "69_Rabbit_Young_Femu_cdf463e6":     ("Femur",            "Rabbit"),
    "113_Normal_Lung_113_19bc7ae7":      ("Lung",             "Human"),
    "135_Submandib_Gland_2d196d08":      ("Submandib. gland", "Human"),
    "406_Human_SemVesicle_b1cd217d":     ("Seminal vesicle",  "Human"),
    "416_Human_FallTube_-_af2fa66e":     ("Fallopian tube",   "Human"),
}

# ── load all tile-level results across batches ──────────────────────
results_dir = pathlib.Path(r"../../final_results")
batch_files = [
    "eval_metrics_batch_1_20260309_142143.json",
    "eval_metrics_batch_2_20260309_142834.json",
    "eval_metrics_batch_3_20260309_142851.json",
]

rows = []
for bf in batch_files:
    with open(results_dir / bf) as f:
        data = json.load(f)
    for tile in data["tile_metrics"]:
        tissue, species = SLIDE_META[tile["slide_id"]]
        rows.append({
            "tissue_type": tissue,
            "species": species,
            "slide_id": tile["slide_id"],
            "n_cells": tile["n_cells"],
            "accuracy": tile["accuracy"],
        })

tiles_df = pd.DataFrame(rows)

# ── per-tissue weighted accuracy (weighted by n_cells per tile) ─────
def weighted_acc(g):
    return (g["n_cells"] * g["accuracy"]).sum() / g["n_cells"].sum()

tissue_agg = (
    tiles_df
    .groupby(["tissue_type", "species"])
    .agg(
        Cells=("n_cells", "sum"),
        Tiles=("n_cells", "count"),
        Acc=("accuracy", lambda g: weighted_acc(
            tiles_df.loc[g.index, ["n_cells", "accuracy"]]
        )),
    )
    .reset_index()
    .rename(columns={"tissue_type": "Tissue type", "species": "Species"})
)

# ── sort in the same order as the paper table ───────────────────────
tissue_order = list(dict.fromkeys(
    SLIDE_META[sid][0] for sid in [
        "177_Human_Appendix_-_d0dcc529",
        "425_Human_Ovari_-_20_7ee9f54b",
        "76_HO135E_Compact_Bo_84b1d03b",
        "93 Muscle T_S_ H_E - 2012-08-07",
        "190_Human_Pancreas_1_138151f1",
        "215-1_Rat_Kidney_Car_9a27ed5a",
        "396_Human_Prostate_3_d1b5a330",
        "434_Human_Cervix_434_12bce85a",
        "69_Rabbit_Young_Femu_cdf463e6",
        "113_Normal_Lung_113_19bc7ae7",
        "135_Submandib_Gland_2d196d08",
        "406_Human_SemVesicle_b1cd217d",
        "416_Human_FallTube_-_af2fa66e",
    ]
))
tissue_agg["_order"] = tissue_agg["Tissue type"].map(
    {t: i for i, t in enumerate(tissue_order)}
)
tissue_agg = tissue_agg.sort_values("_order").drop(columns="_order")

# ── format accuracy as percentage with one decimal ──────────────────
tissue_agg["Acc. (%)"] = (tissue_agg["Acc"] * 100).round(1)
tissue_agg = tissue_agg.drop(columns="Acc")

# ── append aggregate row ────────────────────────────────────────────
total_cells = tissue_agg["Cells"].sum()
total_tiles = tissue_agg["Tiles"].sum()
total_acc = (tiles_df["n_cells"] * tiles_df["accuracy"]).sum() / tiles_df["n_cells"].sum()

agg_row = pd.DataFrame([{
    "Tissue type": "All tissues",
    "Species": "---",
    "Cells": total_cells,
    "Tiles": total_tiles,
    "Acc. (%)": round(total_acc * 100, 1),
}])
tissue_agg = pd.concat([tissue_agg, agg_row], ignore_index=True)

tissue_agg["Cells"] = tissue_agg["Cells"].astype(int)
tissue_agg["Tiles"] = tissue_agg["Tiles"].astype(int)

tissue_agg